# ViFoodKG: Khám phá đồ thị Neo4j (Exploratory Data Analysis)
Notebook này chứa các hàm tiện ích được thiết kế để dễ dàng tương tác với cấu trúc Knowledge Graph Neo4j hiện tại. Tại đây, bạn có thể thực thi các lệnh đếm tổng số Nodes/Edges, kiểm tra mức độ phủ thông tin (Sparsity) và phân tích các quan hệ (Relations).

In [ ]:
!pip install neo4j python-dotenv pandas tabulate

In [ ]:
import os
import pandas as pd
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Tải biến môi trường từ file .env
load_dotenv()  # Yêu cầu có file .env ngang hàng với thư mục chạy

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")

# Khởi tạo Neo4j Driver
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
print("✅ Kết nối Neo4j thành công!")

## 1. Thống kê Cơ bản (Basic Statistics)
Tính tổng số Nodes và Relationships hiện tại trong toàn bộ đồ thị tri thức.

In [ ]:
def get_basic_stats(tx):
    node_count = tx.run("MATCH (n) RETURN count(n) AS count").single()["count"]
    rel_count = tx.run("MATCH ()-[r]->() RETURN count(r) AS count").single()["count"]
    dish_count = tx.run("MATCH (d:Dish) RETURN count(d) AS count").single()["count"]
    return node_count, rel_count, dish_count

with driver.session() as session:
    nodes, rels, dishes = session.execute_read(get_basic_stats)
    print(f"Tổng số Nodes: {nodes:,}")
    print(f"Tổng số Relationships: {rels:,}")
    print(f"Tổng số Món ăn (Dish nodes): {dishes:,}")

## 2. Đánh giá Mật độ & Độ thưa thớt (Density & Sparsity EDA)
Kiểm tra xem trung bình mỗi món ăn có bao nhiêu quan hệ (Knowledge Triples).

In [ ]:
def get_sparsity_metrics(tx):
    query = """
    MATCH (d:Dish)
    OPTIONAL MATCH (d)-[r]->()
    WITH d, count(r) as rel_count
    RETURN 
        avg(rel_count) AS avg_rels,
        min(rel_count) AS min_rels,
        max(rel_count) AS max_rels,
        sum(CASE WHEN rel_count = 0 THEN 1 ELSE 0 END) AS orphan_count
    """
    result = tx.run(query).single()
    return result

with driver.session() as session:
    metrics = session.execute_read(get_sparsity_metrics)
    print(f"Trung bình Số quan hệ/Món ăn: {metrics['avg_rels']:.2f}")
    print(f"Số quan hệ Nhỏ nhất: {metrics['min_rels']} | Lớn nhất: {metrics['max_rels']}")
    print(f"📌 Có {metrics['orphan_count']} món ăn mồ côi (Không có liên kết nào!)")

## 3. Phân bố Dữ liệu theo Loại Quan Hệ (Relation Coverage)
Kiểm đếm xem Ontology của chúng ta có loại quan hệ nào chiếm đa số và loại nào đang bị thiếu hụt.

In [ ]:
def get_relation_distribution(tx):
    query = """
    MATCH ()-[r]->()
    RETURN type(r) AS RelationshipType, count(r) AS Count
    ORDER BY Count DESC
    """
    return tx.run(query).data()

with driver.session() as session:
    dist_data = session.execute_read(get_relation_distribution)
    df_rel = pd.DataFrame(dist_data)
    from tabulate import tabulate
    print(tabulate(df_rel, headers='keys', tablefmt='psql', showindex=False))

## 4. Phân bố Nguồn dữ liệu (Source Distribution)
Đánh giá xem hệ thống Hybrid LLM đã đóng góp bao nhiêu phần trăm chất xám so với dữ liệu crawling từ Website.

In [ ]:
def get_source_distribution(tx):
    query = """
    MATCH ()-[r]->()
    RETURN r.source_url AS Source, count(r) AS Count
    ORDER BY Count DESC
    LIMIT 5
    """
    return tx.run(query).data()

with driver.session() as session:
    source_data = session.execute_read(get_source_distribution)
    df_source = pd.DataFrame(source_data)
    print(tabulate(df_source, headers='keys', tablefmt='psql', showindex=False))

## 5. Thống kê Vector Embeddings (Vectorization Coverage)
Kiểm tra xem hệ thống đã nhúng (embed) thành công bao nhiêu quan hệ để phục vụ Vector Search (Hybrid Retrieval).

In [ ]:
def get_embedding_stats(tx):
    query = """
    MATCH ()-[r]->()
    RETURN 
        count(r) AS total_rels,
        sum(CASE WHEN r.embedding IS NOT NULL THEN 1 ELSE 0 END) AS embedded_count,
        sum(CASE WHEN r.embedding IS NULL AND r.verbalized_text IS NOT NULL THEN 1 ELSE 0 END) AS missing_embedding_count
    """
    return tx.run(query).single()

with driver.session() as session:
    embed_stats = session.execute_read(get_embedding_stats)
    print(f"Tổng số Quan hệ (Edges): {embed_stats['total_rels']:,}")
    print(f"🔹 Đã nhúng Vector (Có Embedding): {embed_stats['embedded_count']:,}")
    print(f"⚠️ Vùng mù (Có Text nhưng Thiếu Embedding): {embed_stats['missing_embedding_count']:,}")
    coverage = (embed_stats['embedded_count'] / embed_stats['total_rels']) * 100 if embed_stats['total_rels'] > 0 else 0
    print(f"=> Tỷ lệ bao phủ Vector Search: {coverage:.2f}%")


In [ ]:
# Đóng kết nối khi hoàn tất
driver.close()